In [1]:
import scipy
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import rel_entr

# Define P_train and different groupings

In [ ]:
yas = np.array([
    0.95 / 4,  # 0
    0.05 / 4,  # 1
    0.8  / 4,  # 2
    0.2  / 4,  # 3
    0.05 / 4,  # 4
    0.95 / 4,  # 5
    0.2  / 4,  # 6
    0.8  / 4   # 7
])
y_group_indices = [
    [0,1,2,3],
    [4,5,6,7],
]
a_group_indices = [
    [0,2,4,6],
    [1,3,5,7],
]
s_group_indices = [
    [0,1,4,5],
    [2,3,6,7],
]
ay_group_indices = [
    [0, 2],  # Group for w_0
    [1, 3],  # Group for w_1
    [4, 6],  # Group for w_2
    [5, 7],  # Group for w_3
]
sy_group_indices = [
    [0, 1],  # Group for w_0
    [2, 3],  # Group for w_1
    [4, 5],  # Group for w_2
    [6, 7],  # Group for w_3
]
yas_group_indices = [
    [0],
    [1],
    [2],
    [3],
    [4],
    [5],
    [6],
    [7],
]
sc_no_sc_indices = [
    [0,2,5,7],
    [1,3,4,6],
]
random = [
    [0,1,2,3,4,5,6,7]
]
ay_8_group_indices = [
    [0, 2],  # Group for w_0
    [0, 2],  # Group for w_0
    [1, 3],  # Group for w_1
    [1, 3],  # Group for w_1
    [4, 6],  # Group for w_2
    [4, 6],  # Group for w_2
    [5, 7],  # Group for w_3
    [5, 7],  # Group for w_3
]
sy_8_group_indices = [
    [0, 1],  # Group for w_0
    [0, 1],  # Group for w_0
    [2, 3],  # Group for w_1
    [2, 3],  # Group for w_1
    [4, 5],  # Group for w_2
    [4, 5],  # Group for w_2
    [6, 7],  # Group for w_3
    [6, 7],  # Group for w_3
]
indices_dict = {'Y': y_group_indices,'A': a_group_indices,'S': s_group_indices,'AY': ay_group_indices,'SY': sy_group_indices,'YAS': yas_group_indices,'SC/no-SC': sc_no_sc_indices,'Random': random,'AY_8': ay_8_group_indices,'SY_8': sy_8_group_indices}  

In [7]:
print(indices_dict.keys())

dict_keys(['Y', 'A', 'S', 'AY', 'SY', 'YAS', 'SC/no-SC', 'Random', 'AY_8', 'SY_8'])


# Calculate KL divergence for gDRO

In [4]:
# Generic weighted distribution function
def weighted_distribution(yas, group_indices, weights, noise = 0):
    result = np.zeros_like(yas)
    for w, group in zip(weights, group_indices):
        norm = sum(yas[j] for j in group)
        for j in group:
            result[j] += (1-noise)*(w * yas[j] / norm) + noise*yas[j] # when there is noise, add some of the original distirbution

    return result


# KL-divergence objective against uniform target
def kl_objective(yas, group_indices, normalize=True, noise = 0):
    def inner(ws):
        weighted_dist = weighted_distribution(yas, group_indices, ws, noise)
        if normalize: # weights should sum to 1
            weighted_dist /= np.sum(weighted_dist)
            ws /= np.sum(ws)
        target = np.ones_like(weighted_dist) / len(weighted_dist)
        return np.sum(rel_entr(target, weighted_dist))
    return inner

def minimise_kl(yas, group_indices,normalize=True,noise=0):
    x0 = [1/len(group_indices)] * len(group_indices)
    bounds = [(1e-8, None)] * len(group_indices)
    
    # find optimal weights
    result = minimize(kl_objective(yas, group_indices,normalize=normalize,noise=noise), x0, bounds=bounds)
    result.x /= np.sum(result.x) # for some reason not always perfectly normalised

    # calculate dist then KL divergence with those weights
    weighted_dist = weighted_distribution(yas, group_indices, result.x, noise=noise)

    return np.sum(rel_entr(np.ones_like(yas) / len(yas), weighted_dist))

In [24]:
noisy_AY = ['noisy_AY_001','noisy_AY_005','noisy_AY_010','noisy_AY_025','noisy_AY_050']

kl_div_gdro = []
subgroups_calculated = []

for key in indices_dict:
    value = minimise_kl(yas, indices_dict[key], normalize=True, noise=0)
    kl_div_gdro.append(value)
    subgroups_calculated.append(key)

# Loop through noisy AY subgroups
for i,noise in enumerate([0.01, 0.05, 0.10, 0.25, 0.5]):
    value = minimise_kl(yas, indices_dict['AY'], normalize=True, noise=noise)
    kl_div_gdro.append(value)
    subgroups_calculated.append(noisy_AY[i])

kl_gdro_df = pd.DataFrame(kl_div_gdro, index=subgroups_calculated, columns=['KL_divergence'])
kl_gdro_df.index.name = 'Subgroup'
kl_gdro_df.to_csv('processed_results/kl_gdro.csv')
kl_gdro_df

,KL_divergence
Subgroup,
Y,0.526755
A,0.526755
S,0.526755
AY,0.113415
SY,0.526755
YAS,0.000000
SC/no-SC,0.113415
Random,0.526755
AY_8,0.113415


# Calculate KL divergence for resampling

In [25]:
target = [1/8] * 8

kl_div_resampling = []
subgroups_calculated = []

for key in indices_dict:
    weighted_dist = weighted_distribution(yas, indices_dict[key],weights = [1/len(indices_dict[key])]*len(indices_dict[key]), noise=0)
    kl_div_resampling.append(sum(rel_entr(target,weighted_dist)))
    subgroups_calculated.append(key)

# Loop through noisy AY subgroups
for i,noise in enumerate([0.01, 0.05, 0.10, 0.25, 0.5]):
    weighted_dist = weighted_distribution(yas, indices_dict['AY'],weights = [1/len(indices_dict['AY'])]*len(indices_dict['AY']), noise=noise)
    kl_div_resampling.append(sum(rel_entr(target,weighted_dist)))
    subgroups_calculated.append(noisy_AY[i])

kl_resampling_df = pd.DataFrame(kl_div_resampling, index=subgroups_calculated, columns=['KL_divergence'])
kl_resampling_df.index.name = 'Subgroup'
kl_resampling_df.to_csv('processed_results/kl_resampling.csv')
kl_resampling_df

,KL_divergence
Subgroup,
Y,0.526755
A,0.526755
S,0.526755
AY,0.113415
SY,0.526755
YAS,0.000000
SC/no-SC,0.113415
Random,0.526755
AY_8,0.113415
